In [2]:
"""
OpenCV Photo Editor
A CLI tool to apply multiple image processing operations in sequence.
"""

import cv2
import numpy as np
import os


def load_image():
    path = input("Enter path to your image: ").strip()
    if not os.path.exists(path):
        print("❌ File not found. Try again.")
        return None
    img = cv2.imread(path)
    if img is None:
        print("❌ Could not read image (unsupported format?).")
        return None
    print(f"✅ Loaded image: {img.shape[1]}x{img.shape[0]} pixels")
    return img


def show(img, title="Preview"):
    cv2.imshow(title, img)
    print("Press any key on the image window to continue...")
    cv2.waitKey(0)
    cv2.destroyAllWindows()


# ---------- 1. Brightness ----------
def brightness(img):
    val = int(input("Brightness change (-100 to 100): "))
    # beta shifts every pixel value up/down
    return cv2.convertScaleAbs(img, alpha=1, beta=val)


# ---------- 2. Contrast ----------
def contrast(img):
    val = float(input("Contrast factor (0.5 - 3.0, 1.0 = no change): "))
    # alpha scales pixel values around the mean spread
    return cv2.convertScaleAbs(img, alpha=val, beta=0)


# ---------- 3. Crop ----------
def crop(img):
    h, w = img.shape[:2]
    print(f"Image size: {w}x{h}")
    x1 = int(input("x1: ")); y1 = int(input("y1: "))
    x2 = int(input("x2: ")); y2 = int(input("y2: "))
    x1, x2 = sorted((max(0, x1), min(w, x2)))
    y1, y2 = sorted((max(0, y1), min(h, y2)))
    return img[y1:y2, x1:x2]


# ---------- 4. Resize ----------
def resize(img):
    w = int(input("New width: "))
    h = int(input("New height: "))
    return cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)


# ---------- 5. Rotate ----------
def rotate(img):
    angle = float(input("Rotation angle (degrees, +ve = counter-clockwise): "))
    h, w = img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(img, M, (w, h))


# ---------- 6. Beauty Filter ----------
def beauty_filter(img):
    # bilateralFilter smooths flat skin tones but preserves strong edges (eyes, hair)
    return cv2.bilateralFilter(img, d=15, sigmaColor=75, sigmaSpace=75)


# ---------- 7. Sharpen ----------
def sharpen(img):
    kernel = np.array([[0, -1, 0],
                        [-1, 5, -1],
                        [0, -1, 0]])
    return cv2.filter2D(img, -1, kernel)


# ---------- 8. Blur ----------
def blur(img):
    k = int(input("Blur strength (odd number, e.g. 5, 15, 25): "))
    if k % 2 == 0:
        k += 1
    return cv2.GaussianBlur(img, (k, k), 0)


# ---------- 9. Remove Noise ----------
def remove_noise(img):
    # Non-Local Means: compares patches across the image, not just neighboring pixels
    return cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)


# ---------- 10. Black & White ----------
def black_white(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)  # keep 3 channels for consistency


# ---------- 11. Cartoon Effect ----------
def cartoon(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_blur = cv2.medianBlur(gray, 5)
    edges = cv2.adaptiveThreshold(
        gray_blur, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY, blockSize=9, C=9
    )
    color = cv2.bilateralFilter(img, d=9, sigmaColor=250, sigmaSpace=250)
    cartoon_img = cv2.bitwise_and(color, color, mask=edges)
    return cartoon_img


# ---------- 12. Pencil Sketch ----------
def pencil_sketch(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    inverted = 255 - gray
    blurred = cv2.GaussianBlur(inverted, (21, 21), 0)
    inverted_blur = 255 - blurred
    sketch = cv2.divide(gray, inverted_blur, scale=256.0)
    return cv2.cvtColor(sketch, cv2.COLOR_GRAY2BGR)


# ---------- 13. Edge Detection ----------
def edge_detection(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)


# ---------- 14. Background Blur ----------
def background_blur(img):
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5)

    if len(faces) == 0:
        print("⚠️ No face detected, blurring edges of frame instead.")
        h, w = img.shape[:2]
        faces = [(w // 4, h // 4, w // 2, h // 2)]  # fallback: center rectangle

    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    for (x, y, w, h) in faces:
        pad = int(0.3 * w)
        cv2.rectangle(mask, (max(0, x - pad), max(0, y - pad)),
                       (x + w + pad, y + h + pad), 255, -1)

    blurred_bg = cv2.GaussianBlur(img, (35, 35), 0)
    mask_3ch = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    result = np.where(mask_3ch == 255, img, blurred_bg)
    return result.astype(np.uint8)


# ---------- 15. HDR Effect ----------
def hdr_effect(img):
    # detailEnhance boosts local contrast/detail while smoothing flat areas
    return cv2.detailEnhance(img, sigma_s=12, sigma_r=0.15)


# ---------- 16. Save Image ----------
def save_image(img):
    path = input("Enter output filename (e.g. output.jpg): ").strip()
    cv2.imwrite(path, img)
    print(f"✅ Saved to {path}")


# ---------- Main Menu Loop ----------
def main():
    print("===== OpenCV Photo Editor =====")
    img = None
    while img is None:
        img = load_image()

    working = img.copy()

    menu = """
1. Brightness Enhancement
2. Contrast Enhancement
3. Crop Image
4. Resize Image
5. Rotate Image
6. Beauty Filter (Skin Smoothing)
7. Sharpen Image
8. Blur Image
9. Remove Noise
10. Black & White
11. Cartoon Effect
12. Pencil Sketch
13. Edge Detection
14. Background Blur
15. HDR Effect
16. Save Image
17. Exit
"""
    operations = {
        "1": brightness, "2": contrast, "3": crop, "4": resize,
        "5": rotate, "6": beauty_filter, "7": sharpen, "8": blur,
        "9": remove_noise, "10": black_white, "11": cartoon,
        "12": pencil_sketch, "13": edge_detection,
        "14": background_blur, "15": hdr_effect,
    }

    while True:
        print(menu)
        choice = input("Enter your choice: ").strip()

        if choice == "17":
            print("Goodbye! 👋")
            break
        elif choice == "16":
            save_image(working)
        elif choice in operations:
            try:
                working = operations[choice](working)
                show(working, title=f"After operation {choice}")
            except Exception as e:
                print(f"⚠️ Error: {e}")
        else:
            print("❌ Invalid choice, try again.")

    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

===== OpenCV Photo Editor =====


Enter path to your image:  ../Img/women.jpeg


✅ Loaded image: 1080x1000 pixels

1. Brightness Enhancement
2. Contrast Enhancement
3. Crop Image
4. Resize Image
5. Rotate Image
6. Beauty Filter (Skin Smoothing)
7. Sharpen Image
8. Blur Image
9. Remove Noise
10. Black & White
11. Cartoon Effect
12. Pencil Sketch
13. Edge Detection
14. Background Blur
15. HDR Effect
16. Save Image
17. Exit



Enter your choice:  ../Img/women.jpeg


❌ Invalid choice, try again.

1. Brightness Enhancement
2. Contrast Enhancement
3. Crop Image
4. Resize Image
5. Rotate Image
6. Beauty Filter (Skin Smoothing)
7. Sharpen Image
8. Blur Image
9. Remove Noise
10. Black & White
11. Cartoon Effect
12. Pencil Sketch
13. Edge Detection
14. Background Blur
15. HDR Effect
16. Save Image
17. Exit



...

Press any key on the image window to continue...

1. Brightness Enhancement
2. Contrast Enhancement
3. Crop Image
4. Resize Image
5. Rotate Image
6. Beauty Filter (Skin Smoothing)
7. Sharpen Image
8. Blur Image
9. Remove Noise
10. Black & White
11. Cartoon Effect
12. Pencil Sketch
13. Edge Detection
14. Background Blur
15. HDR Effect
16. Save Image
17. Exit



Enter your choice:  12


Press any key on the image window to continue...

1. Brightness Enhancement
2. Contrast Enhancement
3. Crop Image
4. Resize Image
5. Rotate Image
6. Beauty Filter (Skin Smoothing)
7. Sharpen Image
8. Blur Image
9. Remove Noise
10. Black & White
11. Cartoon Effect
12. Pencil Sketch
13. Edge Detection
14. Background Blur
15. HDR Effect
16. Save Image
17. Exit



Enter your choice:  17


Goodbye! 👋
